# phase 1 :- data collection

In [1]:
import yfinance as yf
import pandas as pd

stock = "AAPL"

data = yf.download(stock, start="2019-01-01", end="2026-05-04")

# Fix column format
data.columns = data.columns.droplevel(1)

# Save CSV
data.to_csv("stock_data.csv")

print(data.tail())

/tmp/ipykernel_3754/1024425200.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(stock, start="2019-01-01", end="2026-05-04")
[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Date                                                                
2026-04-27  267.609985  268.359985  265.070007  266.089996  41466800
2026-04-28  270.709991  273.230011  268.660004  272.339996  40018900
2026-04-29  270.170013  271.040009  267.040009  267.549988  30047900
2026-04-30  271.350006  276.000000  268.140015  270.500000  91848200
2026-05-01  280.140015  287.220001  278.369995  278.859985  79915400


###convert into the csv

In [2]:
data.to_csv("stock_data.csv")

### download the data

In [3]:
from google.colab import files
files.download("stock_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# phase 2 :- data strorage

In [5]:
import pandas as pd

data = pd.read_csv("data/data.csv")
print(data.head())

         Date      Close       High        Low       Open     Volume
0  2019-01-02  37.503727  37.724590  36.627404  36.784146  148158800
1  2019-01-03  33.768078  34.606402  33.722955  34.193175  365248800
2  2019-01-04  35.209618  35.278490  34.150434  34.323797  234428400
3  2019-01-07  35.131245  35.344984  34.649149  35.314110  219111200
4  2019-01-08  35.800949  36.055060  35.271357  35.518341  164101200


###  fix date columns

In [6]:
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)

###sort data (why  :- old to new)

In [7]:
data.sort_index(inplace=True)

 ### data validation

In [8]:
print(data.info())
print(data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1843 entries, 2019-01-02 to 2026-05-01
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   1843 non-null   float64
 1   High    1843 non-null   float64
 2   Low     1843 non-null   float64
 3   Open    1843 non-null   float64
 4   Volume  1843 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 86.4 KB
None
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


### data order

In [9]:
data = data[['Open', 'High', 'Low', 'Close', 'Volume']]

###save clean version


In [10]:
data.to_csv("clean_stock_data.csv")

In [12]:
data.to_csv("data/clean_stock_data.csv", index=False)

#phase 3 :- data enggnering

###create a target variable

In [ ]:
data['Target'] = data['Close'].shift(-1)
data.dropna(inplace=True)
print(data['Target'].tail() )


###create the features


#### moving average

In [ ]:
data['MA_10'] = data['Close'].rolling(window=10).mean()
data['MA_50'] = data['Close'].rolling(window=50).mean()

#### daily return

In [ ]:
data['Return'] = data['Close'].pct_change()

####Volatility (Risk)

In [ ]:
data['Volatility'] = data['Return'].rolling(window=10).std()

#### clean data after using rolling fucntion

In [ ]:
data.dropna(inplace=True)

#### define feature look like x and y

In [ ]:
X = data[['Open', 'High', 'Low', 'Close', 'Volume', 'MA_10', 'MA_50', 'Return', 'Volatility']]

In [ ]:
y = data['Target']

In [ ]:
print(data.tail())

# phase 4 ;- data modeling


### step1 train - split ( time-based)

In [ ]:
train_size = int(len(X) * 0.8)

X_train = X[:train_size]
X_test = X[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]

### first model ( linear regression)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.show()

#### quick test

In [ ]:
import numpy as np
baseline = X_test['Close']

from sklearn.metrics import mean_absolute_error
print("Baseline MAE:", mean_absolute_error(y_test, baseline))

Baseline MAE ≈ 2.83  
Model MAE ≈ 2.88
this very similiar so model

### second model ( random forest)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error

mae_rf = mean_absolute_error(y_test, y_pred_rf)
print("Random Forest MAE (with Close):", mae_rf)

In [ ]:
X_no_close = data[['Open','High','Low','Volume','MA_10','MA_50','Return','Volatility']]

# Split again
train_size = int(len(X_no_close) * 0.8)

X_train_nc = X_no_close[:train_size]
X_test_nc = X_no_close[train_size:]

y_train_nc = y[:train_size]
y_test_nc = y[train_size:]

# Train
model_rf_nc = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf_nc.fit(X_train_nc, y_train_nc)

y_pred_rf_nc = model_rf_nc.predict(X_test_nc)

mae_rf_nc = mean_absolute_error(y_test_nc, y_pred_rf_nc)
print("Random Forest MAE (without Close):", mae_rf_nc)